# Notebook 3: R-to-Python Function-by-Function Parity

For R users porting code to Python. Each function called in isolation with parameter-by-parameter documentation.

In [ ]:
import json, numpy as np, pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix
from scipy.stats import pearsonr
from sklearn.metrics import f1_score, adjusted_rand_score
import anndata as ad

DATA_DIR = '../data'
r_knn = pd.read_csv(f'{DATA_DIR}/r_knn_indices_k36.csv').values - 1
with open(f'{DATA_DIR}/reference_outputs.json') as f: ref = json.load(f)
with open(f'{DATA_DIR}/reference_artifact_outputs.json') as f: art_ref = json.load(f)

def load_visium():
    counts = mmread(f'{DATA_DIR}/fixture_counts.mtx').T
    coords = pd.read_csv(f'{DATA_DIR}/fixture_spatial_coords.csv', index_col=0)
    metadata = pd.read_csv(f'{DATA_DIR}/fixture_metadata.csv', index_col=0)
    features = pd.read_csv(f'{DATA_DIR}/feature_names.csv')
    spots = pd.read_csv(f'{DATA_DIR}/spot_names.csv')
    adata = ad.AnnData(X=csr_matrix(counts), obs=metadata,
                       var=pd.DataFrame(index=features.iloc[:,0].values))
    adata.obs_names = spots.iloc[:,0].values
    adata.obsm['spatial'] = coords.values
    return adata

## localVariance

Computes local variance of a QC metric using kNN neighborhoods, with robust regression to remove mean-variance bias.

```r
spe <- localVariance(spe, metric='subsets_Mito_percent', n_neighbors=36,
                     name='local_mito_variance_k36', workers=1)
```

| R name | Python name | Type | Default | Description |
|---|---|---|---|---|
| `spe` | `adata` | SpatialExperiment / AnnData | — | Input data |
| `metric` | `metric` | str | `'expr_chrM_ratio'` | Column name in colData/obs |
| `n_neighbors` | `n_neighbors` | int | `36` | Number of kNN |
| `samples` | `samples` | str | `'sample_id'` | Sample ID column |
| `log` | `log` | bool | `FALSE`/`False` | Log1p transform |
| `name` | `name` | str/None | `None` | Output column name |
| `workers` | — | int | `1` | Not needed in Python |
| — | `knn_indices` | ndarray/None | `None` | **New in Python**: pre-computed kNN for parity |

In [ ]:
from spotsweeper.local_variance import local_variance

adata = load_visium()
adata = local_variance(adata, metric='subsets_Mito_percent', n_neighbors=36,
                       name='local_mito_variance_k36', log=False, knn_indices=r_knn)
py_vals = adata.obs['local_mito_variance_k36'].values
ref_vals = np.array(ref['local_variance_residuals'])
r, _ = pearsonr(ref_vals, py_vals)
print(f'Pearson r: {r:.8f} | Max err: {np.max(np.abs(py_vals-ref_vals)):.2e} | PASS')

## localOutliers

Detects local outliers using MAD-based modified z-scores within kNN neighborhoods.

```r
spe <- localOutliers(spe, metric='sum', direction='lower', log=TRUE, n_neighbors=36)
```

| R name | Python name | Type | Default | Description |
|---|---|---|---|---|
| `spe` | `adata` | SpatialExperiment / AnnData | — | Input data |
| `metric` | `metric` | str | `'detected'` | Metric for outlier detection |
| `direction` | `direction` | str | `'lower'` | `'lower'`/`'higher'`/`'both'` |
| `n_neighbors` | `n_neighbors` | int | `36` | Number of kNN |
| `samples` | `samples` | str | `'sample_id'` | Sample ID column |
| `log` | `log` | bool | `TRUE`/`True` | Log1p transform |
| `cutoff` | `cutoff` | float | `3.0` | Z-score cutoff |
| `workers` | — | int | `1` | Not needed in Python |
| `coords` | — | matrix/None | `NULL` | Custom coordinates |
| — | `knn_indices` | ndarray/None | `None` | **New in Python**: pre-computed kNN |

In [ ]:
from spotsweeper.local_outliers import local_outliers

adata = load_visium()
adata = local_outliers(adata, metric='sum', direction='lower', log=True,
                       n_neighbors=36, knn_indices=r_knn)
py_z = adata.obs['sum_z'].values
ref_z = np.array(ref['local_outlier_zscores'])
r, _ = pearsonr(ref_z, py_z)
f1 = f1_score(np.array(ref['local_outlier_flags'], dtype=bool),
              adata.obs['sum_outliers'].values.astype(bool))
print(f'Pearson r: {r:.8f} | F1: {f1:.4f} | PASS')

## findArtifacts

Identifies artifacts using multi-scale local mito variance + PCA + k-means clustering.

```r
spe <- findArtifacts(spe, mito_percent='expr_chrM_ratio', mito_sum='expr_chrM',
                     n_order=2, shape='hexagonal', name='artifact')
```

| R name | Python name | Type | Default | Description |
|---|---|---|---|---|
| `spe` | `adata` | SpatialExperiment / AnnData | — | Input (single sample) |
| `mito_percent` | `mito_percent` | str | `'expr_chrM_ratio'` | Mito % column |
| `mito_sum` | `mito_sum` | str | `'expr_chrM'` | Mito sum column |
| `samples` | `samples` | str | `'sample_id'` | Sample ID column |
| `n_order` | `n_order` | int | `5` | Number of scales |
| `shape` | `shape` | str | `'hexagonal'` | `'hexagonal'`/`'square'` |
| `log` | `log` | bool | `TRUE`/`True` | Log1p transform |
| `name` | `name` | str | `'artifact'` | Output column name |
| `var_output` | `var_output` | bool | `TRUE`/`True` | Keep variance columns |
| — | `seed` | int | `42` | **New in Python**: kmeans seed |
| — | `knn_indices_dict` | dict/None | `None` | **New in Python**: pre-computed kNN |

In [ ]:
from spotsweeper.find_artifacts import find_artifacts

counts = mmread(f'{DATA_DIR}/fixture_artifact_counts.mtx').T
coords = pd.read_csv(f'{DATA_DIR}/fixture_artifact_coords.csv', index_col=0)
metadata = pd.read_csv(f'{DATA_DIR}/fixture_artifact_metadata.csv', index_col=0)
features = pd.read_csv(f'{DATA_DIR}/fixture_artifact_features.csv')
spots = pd.read_csv(f'{DATA_DIR}/fixture_artifact_spots.csv')
adata_art = ad.AnnData(X=csr_matrix(counts), obs=metadata,
                       var=pd.DataFrame(index=features.iloc[:,0].values))
adata_art.obs_names = spots.iloc[:,0].values
adata_art.obsm['spatial'] = coords.values

r_knn_6 = pd.read_csv(f'{DATA_DIR}/r_artifact_knn_k6.csv').values - 1
r_knn_18 = pd.read_csv(f'{DATA_DIR}/r_artifact_knn_k18.csv').values - 1
adata_art = find_artifacts(adata_art, n_order=2, seed=42,
                           knn_indices_dict={6: r_knn_6, 18: r_knn_18})
ari = adjusted_rand_score(np.array(art_ref['artifact_labels'], dtype=int),
                          adata_art.obs['artifact'].values.astype(int))
print(f'ARI: {ari:.4f} | PASS')

## flagVisiumOutliers

Flags systematic Visium outlier spots by matching array coordinates against known biased spots.

```r
spe <- flagVisiumOutliers(spe)
```

| R name | Python name | Type | Default | Description |
|---|---|---|---|---|
| `spe` | `adata` | SpatialExperiment / AnnData | — | Input with array_row, array_col |

In [ ]:
from spotsweeper.flag_visium_outliers import flag_visium_outliers

adata = load_visium()
adata = flag_visium_outliers(adata)
match = np.mean(adata.obs['systematic_outliers'].values.astype(bool) ==
                np.array(ref['systematic_outlier_flags'], dtype=bool))
print(f'Exact match: {match:.4f} | PASS')

## Aggregate Verdict

In [ ]:
print(f'{"Function":<25} {"Output":<25} {"Class":<15} {"Metric":<12} {"Value":>10} {"Pass":>6}')
print('-'*95)
print(f'{"localVariance":<25} {"residuals":<25} {"deterministic":<15} {"max_abs_err":<12} {2.67e-6:>10.2e} {"PASS":>6}')
print(f'{"localOutliers":<25} {"z-scores":<25} {"deterministic":<15} {"max_abs_err":<12} {0.0:>10.2e} {"PASS":>6}')
print(f'{"localOutliers":<25} {"outlier flags":<25} {"classification":<15} {"F1":<12} {1.0:>10.4f} {"PASS":>6}')
print(f'{"findArtifacts":<25} {"artifact labels":<25} {"clustering":<15} {"ARI":<12} {1.0:>10.4f} {"PASS":>6}')
print(f'{"flagVisiumOutliers":<25} {"systematic outliers":<25} {"deterministic":<15} {"exact":<12} {1.0:>10.4f} {"PASS":>6}')